# 09. Interventional Fidelity

ScorecardModel의 개입적 충실도를 평가한다.

**대상**: 감점요소(adverse feature)에 대해, 인접 bin으로 이동 시:
- 스코어카드 예측 개선(δ_s)과 원모형 실제 개선(δ_b) 비교

**지표**:
- DA (Directional Accuracy): 방향 정확도
- Spearman ρ: 예측-실제 개선량 순위 상관
- IR (Improvement Ratio): δ_b/δ_s
- ENI (Effort-Normalized Improvement): 표준화 노력 대비 실제 개선
- PIR (Penalized Improvement Ratio): 노력 페널티 적용 IR

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate

DATA_DIR = '../.data'

In [2]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }

In [ ]:
def compute_sic_scorecard(scorecard_model, base_model, X_eval, feature_stds,
                           max_search=None):
    """Interventional fidelity: scorecard-based.
    
    For each (sample, adverse feature):
    1. Find current bin
    2. Search for improving bin (adjacent first, expand if needed)
    3. Move x to closest edge of improving bin (minimum effort)
    4. Compare scorecard δ_s vs base model δ_b
    
    Returns dict with DA, Spearman, IR, ENI, PIR, n_pairs
    """
    X_arr = np.asarray(X_eval)
    sm = scorecard_model
    
    # Base model logit on original data: logit(1-prob) = higher is better
    base_logit = logit(1 - base_model.predict_proba(X_arr)[:, 1])
    
    # Scorecard contributions (score space: higher=better, negative=adverse)
    contribs = sm.contributions(X_eval)
    
    delta_s_list = []  # scorecard predicted improvement
    delta_b_list = []  # base model actual improvement
    delta_x_norm_list = []  # |delta_x| / std_x
    
    for feat in sm.features:
        j = feat.index
        fname = feat.name
        bins = feat.bins
        n_bins = len(bins)
        if n_bins < 2:
            continue
        
        std_x = feature_stds.get(fname, 1.0)
        if std_x < 1e-10:
            continue
        
        search_range = max_search or n_bins
        
        for i in range(len(X_arr)):
            # Only adverse features (contrib < 0 in score space)
            if contribs[i, j] >= 0:
                continue
            
            x_val = X_arr[i, j]
            
            # Find current bin
            current_bin_idx = None
            current_score = None
            for b_idx, br in enumerate(bins):
                if br.contains(np.array([x_val]))[0]:
                    current_bin_idx = b_idx
                    current_score = br.score
                    break
            if current_bin_idx is None:
                continue
            
            # Search for improving bin (score > current_score)
            best_target = None
            best_delta_x = float('inf')
            
            for distance in range(1, search_range + 1):
                for direction in [-1, 1]:
                    target_idx = current_bin_idx + direction * distance
                    if target_idx < 0 or target_idx >= n_bins:
                        continue
                    target_bin = bins[target_idx]
                    if target_bin.score <= current_score:
                        continue
                    
                    if direction == 1:
                        x_new = target_bin.lower
                    else:
                        x_new = np.nextafter(target_bin.upper, -np.inf)
                    
                    if np.isinf(x_new):
                        continue
                    
                    dx = abs(x_new - x_val)
                    if dx < best_delta_x:
                        best_delta_x = dx
                        best_target = (target_idx, target_bin, x_new)
                
                if best_target is not None:
                    break
            
            if best_target is None:
                continue
            
            target_idx, target_bin, x_new = best_target
            delta_s = target_bin.score - current_score
            if delta_s <= 1e-8:
                continue
            
            # Base model actual improvement
            x_modified = X_arr[i].copy()
            x_modified[j] = x_new
            new_logit = logit(
                1 - base_model.predict_proba(x_modified.reshape(1, -1))[:, 1]
            )[0]
            delta_b = new_logit - base_logit[i]
            
            delta_x_normalized = abs(x_new - x_val) / std_x
            
            delta_s_list.append(delta_s)
            delta_b_list.append(delta_b)
            delta_x_norm_list.append(delta_x_normalized)
    
    if len(delta_s_list) < 3:
        return {'DA': np.nan, 'Spearman': np.nan, 'IR': np.nan,
                'ENI': np.nan, 'PIR': np.nan, 'n_pairs': len(delta_s_list)}
    
    ds = np.array(delta_s_list)
    db = np.array(delta_b_list)
    dx_norm = np.array(delta_x_norm_list)
    
    da = float(np.mean(db > 0))
    pearson_r = float(np.corrcoef(ds, db)[0, 1])
    sp_rho = float(spearmanr(ds, db)[0])
    ir = float(np.mean(db / ds))
    
    # ENI: Effort-Normalized Improvement
    valid_eni = dx_norm > 0.01  # robust lower bound (was 1e-10)
    eni = float(np.mean(db[valid_eni] / dx_norm[valid_eni])) if valid_eni.sum() > 0 else np.nan
    
    # PIR: Penalized Improvement Ratio
    pir = float(np.mean((db / ds) * np.exp(-dx_norm)))
    
    return {
        'DA': round(da, 4),
        'Pearson_r': round(pearson_r, 4),
        'Spearman': round(sp_rho, 4),
        'IR': round(ir, 4),
        'ENI': round(eni, 4),
        'PIR': round(pir, 4),
        'n_pairs': len(ds),
        'mean_effort': round(float(dx_norm.mean()), 4),
    }

## 실험: Tree depth=1 ScorecardModel

In [4]:
%%time
for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val = t['y_logit_tr'], t['y_logit_val']
    bm = base_models[data_flag]['lgb']
    
    # Feature stds from training data
    feature_stds = {col: float(X_tr[col].std()) for col in X_tr.columns}
    
    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')
    
    # N_SIC: subsample for speed (pred_proba per sample is slow)
    N_SIC = min(500, len(X_test))
    np.random.seed(317)
    sic_idx = np.random.choice(len(X_test), N_SIC, replace=False)
    X_sic = X_test.iloc[sic_idx] if hasattr(X_test, 'iloc') else X_test[sic_idx]
    
    for mono_mode, mono_label in [('none', 'no mono'), ('auto', '+mono')]:
        surr = TreeSurrogate(max_depth=1, monotone_detect_mode=mono_mode, verbose=0)
        surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        
        # ScorecardModel
        sm = surr.to_scorecard_model(
            X_tr, feature_names=list(X_tr.columns),
            max_bins_per_feature=10, min_bin_ratio=0.01)
        
        print(f'\n  === Tree-d1 {mono_label} → SC(max=10, 1%) ===')
        print(f'    {sm}')
        
        result = compute_sic_scorecard(sm, bm, X_sic, feature_stds)
        print(f'    SIC: {result}')


################################################################################
  GMSC
################################################################################



  === Tree-d1 no mono → SC(max=10, 1%) ===
    ScorecardModel(base=3.4572, 10 features, 69 bins)


    SIC: {'DC': 0.7143, 'Pearson_r': 0.686, 'Spearman': 0.4186, 'IR': 0.893, 'ENI': 709.5761, 'PIR': 0.6679, 'n_pairs': 1708, 'mean_effort': 0.381}



  === Tree-d1 +mono → SC(max=10, 1%) ===
    ScorecardModel(base=3.4572, 10 features, 65 bins)


    SIC: {'DC': 0.6654, 'Pearson_r': 0.6945, 'Spearman': 0.4907, 'IR': -0.2209, 'ENI': 422.7436, 'PIR': 0.5765, 'n_pairs': 1584, 'mean_effort': 0.4828}

################################################################################
  HC
################################################################################



  === Tree-d1 no mono → SC(max=10, 1%) ===
    ScorecardModel(base=2.7759, 27 features, 131 bins)


    SIC: {'DC': 0.7153, 'Pearson_r': 0.4592, 'Spearman': 0.3523, 'IR': 3.0239, 'ENI': 12.3275, 'PIR': 1.3853, 'n_pairs': 8054, 'mean_effort': 1.0365}



  === Tree-d1 +mono → SC(max=10, 1%) ===
    ScorecardModel(base=2.7759, 29 features, 131 bins)


    SIC: {'DC': 0.6647, 'Pearson_r': 0.2153, 'Spearman': 0.2551, 'IR': 1.7934, 'ENI': 15.8165, 'PIR': 1.7503, 'n_pairs': 8190, 'mean_effort': 1.0781}
CPU times: total: 8min 4s
Wall time: 32 s


## 지표 해석

| 지표 | 의미 | 이상적 값 |
|---|---|---|
| DA | 스코어카드가 "개선된다"고 한 방향이 실제로 맞는 비율 (Directional Accuracy) | 1.0 |
| Spearman ρ | 예측 개선량과 실제 개선량의 순위 상관 | 1.0 |
| IR | δ_b/δ_s — 1이면 스코어카드 예측이 정확 | 1.0 |
| ENI | 표준화 노력 1단위당 실제 개선량 | 높을수록 효율적 |
| PIR | 노력이 클수록 IR에 페널티 | 높을수록 효율적 |
| mean_effort | 평균 표준화 이동량 (|Δx|/std) | 낮을수록 현실적 |